# Sampling Methods

In [ ]:
# 1. Balanced Sampling
# sample d from [2, D]
# then sample number1 & number2 from [10^(d-1), 10^d - 1] (i.e. from any possible integer of length d)
# then compute answer (we don't randomly sample plus/minus: these are two separate tasks)
import os
import numpy as np
import random
from google.colab import drive

# Mount to drive
drive.mount("/content/drive", force_remount=True)
directory = '/content/drive/MyDrive/CS4782FinalProject/'
if not os.path.exists(directory):
  os.makedirs(directory)

def balanced_sample(D):
  no_of_digits = random.randint(2, D)
  number1 = random.randint(10**(no_of_digits - 1), 10**no_of_digits - 1)
  number2 = random.randint(10**(no_of_digits - 1), 10**no_of_digits - 1)
  return [number1, number2]


# 2. Random Sampling
# sample independently from any possible number [0, 10^D - 1] (up to a predefined maximum number length)

def random_sample(D):
  number1 = random.randint(0, 10**D - 1)
  number2 = random.randint(0, 10**D - 1)
  return [number1, number2]

In [ ]:
pip install num2words

# Representation Functions

In [4]:
from num2words import num2words

def representation(orthography: str, number: int, D = None):

  ''' returns string of new representation of inputted number'''

  number = str(number)

  if orthography == 'decimal':
    return number

  if orthography == 'character':
    if len(number) <= 1:
      return number
    else:
      #number = list(number)
      number = ' '.join(number)
      return number

  elif orthography == 'fixed_character':
    number = str(number)

    # handle negative sign
    signal = ''
    if number[0] == '-':
        signal = '-'
        number = number[1:]

    # pad with zeros
    number = number.zfill(D)

    # add sign back
    if signal:
        number = signal + number

    return ' '.join(number)

  elif orthography == 'underscore':
    return '_'.join(number)

  elif orthography == 'words':
    return num2words(int(number))

  elif orthography == '10based':
    # handle negative sign
    signal = None
    if number[0] == '-':
        signal = '-'
        number = number[1:]

    output = []

    # iterate from least significant digit
    for i, digit in enumerate(number[::-1]):
        if i > 0:
            output.append('1' + '0'*i)
        output.append(digit)

    if signal:
        output.append(signal)

    output = output[::-1]

    return ' '.join(output)

  elif orthography == '10ebased':
    # handle negative sign
    signal = None
    if number[0] == '-':
        signal = '-'
        number = number[1:]

    output = []

    # iterate from least significant digit
    for i, digit in enumerate(number[::-1]):
        output.append('10e' + str(i))
        output.append(digit)

    if signal:
        output.append(signal)

    output = output[::-1]

    return ' '.join(output)

  elif orthography == '10e_underscore':
    # handle negative sign
    signal = None
    if number[0] == '-':
        signal = '-'
        number = number[1:]

    output = []

    # iterate from least significant digit
    for i, digit in enumerate(number[::-1]):
        output.append('10e' + str(i))
        output.append(digit)

    if signal:
        output.append(signal)

    output = output[::-1]

    return '_'.join(output)

  elif orthography == '10e_fixed':
    # handle negative sign
    signal = None
    if number[0] == '-':
        signal = '-'
        number = number[1:]

    output = []

    # iterate from least significant digit
    for i, digit in enumerate(number[::-1]):
        output.append('10e' + str(i))
        output.append(digit)

    if len(number) < D:
      for i in range(len(number), D):
        output.append('0 10e' + str(i))

    if signal:
        output.append(signal)

    output = output[::-1]

    return ' '.join(output)

  else:
    raise NameError('Error! Representation type not found')

# Dataset Generation Functions

In [5]:
def create_dataset(sampling_type: str, representation_type: str, operation: str, D, dim=10000):
  ''' operation: 'plus' or 'minus'
      each element of output is a tuple of: ('arithmetic task: str', 'label: int')
  '''
  dataset = []

  for i in range(dim):
      if sampling_type == 'balanced':
          numbers = balanced_sample(D)
      elif sampling_type == 'random':
          numbers = random_sample(D)
      else:
          raise NameError('Error! Sampling type not found')

      x1 = representation(representation_type, numbers[0], D)
      x2 = representation(representation_type, numbers[1], D)

      question = f'What is {x1} {operation} {x2}?'

      if operation == 'plus':
          result = numbers[0] + numbers[1]
      elif operation == 'minus':
          result = numbers[0] - numbers[1]
      elif operation == 'times':
          result = numbers[0] * numbers[1]
      elif operation == 'divided by':
          result = numbers[0] // numbers[1]
      else:
          raise NameError('Error! Operation not found')

      label = representation(representation_type, result, D)

      dataset.append((question, label))

  return dataset

In [ ]:
# Generate dataset csv files for all types
import pandas as pd
import os

data_dir = os.path.join(directory, "data")

def generate_dataset(D_max, train_samples=20000, eval_samples=200, test_samples=1000):
    datatypes = ['decimal', 'character', 'fixed_character', 'underscore', 'words', '10based', '10ebased']
    operations = ['times']

    for datatype in datatypes:
        for operation in operations:
            # Training and eval data — balanced sampling, large
            print(f"Generating TRAIN dataset for {datatype} {operation}")
            raw_train = create_dataset('balanced', datatype, operation, D_max, train_samples)
            df = pd.DataFrame(raw_train)
            file_name = f"arithmetic_{datatype.replace('-', '_')}_{operation}_{D_max}.csv"
            df.to_csv(os.path.join(data_dir, file_name), index=False)

            print(f"Generating EVAL dataset for {datatype} {operation}")
            raw_eval = create_dataset('balanced', datatype, operation, D_max, eval_samples)
            df = pd.DataFrame(raw_eval)
            file_name = f"arithmetic_eval_{datatype}_{operation}_{D_max}.csv"
            df.to_csv(os.path.join(data_dir, file_name), index=False)

            # Test data — random sampling
            print(f"Generating TEST dataset for {datatype} {operation}")
            raw_test = create_dataset('random', datatype, operation, D_max, test_samples)
            df = pd.DataFrame(raw_test)
            file_name = f"arithmetic_test_{datatype.replace('-', '_')}_{operation}_{D_max}.csv"
            df.to_csv(os.path.join(data_dir, file_name), index=False)

generate_dataset(D_max=5, train_samples=20000, eval_samples=200, test_samples=1000)

In [ ]:
pip install transformers sentencepiece datasets

# Load datasets

In [ ]:
from datasets import load_dataset
import os

datatype = "10ebased"
operation = "plus"
D_max = 5

train_path = os.path.join(directory, f"data/arithmetic_{datatype}_{operation}_{D_max}.csv")
eval_path = os.path.join(directory, f"data/arithmetic_eval_{datatype}_{operation}_{D_max}.csv")
test_path = os.path.join(directory, f"data/arithmetic_test_{datatype}_{operation}_{D_max}.csv")

dataset = {
    "train": load_dataset("csv", data_files=train_path)["train"],
    "eval": load_dataset("csv", data_files=eval_path)["train"],
    "test": load_dataset("csv", data_files=test_path)["train"],
}

# Tokenizer

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model_name = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# For t5 we use sentencepiece tokenization

INPUT_COL = '0'
OUTPUT_COL = '1'

MAX_INPUT_LEN = 128
MAX_OUTPUT_LEN = 128

def tokenize(batch):
    inputs = tokenizer(
        text=[str(x) for x in batch[INPUT_COL]],
        max_length=MAX_INPUT_LEN,
        padding=False,
        truncation=True,
        )

    labels = tokenizer(
        text_target=[str(x) for x in batch[OUTPUT_COL]],
        max_length=MAX_OUTPUT_LEN,
        padding=False,
        truncation=True,
        )

    inputs["labels"] = labels["input_ids"]

    return inputs

# Tokenize Dataset

In [ ]:
from datasets import DatasetDict

dataset = DatasetDict(dataset)
tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)

# Training

In [ ]:
import numpy as np
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# compute_metrics for best checkpoint selection during training
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]


    # Replace invalid -100 values before decoding (both predictions AND labels)
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    def normalize_text(s):
        return " ".join(s.strip().lower().split())

    exact_matches = [
        normalize_text(p) == normalize_text(l)
        for p, l in zip(decoded_preds, decoded_labels)
    ]

    return {"exact_match": sum(exact_matches) / len(exact_matches)}

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=20,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=1e-5,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1570, # show data halfway
    per_device_eval_batch_size=16,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="exact_match",
    greater_is_better=True,
    save_strategy="steps",
    save_steps=1570, # show data halfway
    predict_with_generate=True,
    generation_max_length=MAX_OUTPUT_LEN,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./t5_base_save")
tokenizer.save_pretrained("./t5_base_save")

# Evaluation for Exact Match Accuracy

In [ ]:
import re
import torch
from tqdm import tqdm

def normalize_text(s):
    s = str(s).strip().lower()
    # Fix 10ebased spacing:
    # "110e5" -> "1 10e5"
    # "410e4" -> "4 10e4"
    s = re.sub(r'([0-9])(?=10e\d+)', r'\1 ', s)

    return " ".join(s.split())


def compute_exact_match(model, tokenizer, test_dataset, device="cuda"):
    model.to(device)
    model.eval()

    correct = 0
    total = len(test_dataset)
    results = []

    print(f"Running Exact Match Evaluation on {total} samples...")

    for i in tqdm(range(total)):
        prompt = test_dataset[i]["0"]
        ground_truth = str(test_dataset[i]["1"])

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=MAX_OUTPUT_LEN,
                num_beams=1,
                do_sample=False
            )

        decoded_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        prediction_norm = normalize_text(decoded_text)
        ground_truth_norm = normalize_text(ground_truth)

        is_correct = prediction_norm == ground_truth_norm

        if is_correct:
            correct += 1

        results.append({
            "prompt": prompt,
            "predicted": decoded_text,
            "actual": ground_truth,
            "predicted_normalized": prediction_norm,
            "actual_normalized": ground_truth_norm,
            "is_correct": is_correct
        })

    accuracy = correct / total
    print(f"\nFinal Exact Match Accuracy: {accuracy * 100:.2f}%")
    return accuracy, results


accuracy, detailed_results = compute_exact_match(model, tokenizer, dataset["test"])
print(detailed_results[:10])

# Write Results to CSV

In [ ]:
import csv

file_path = "/content/drive/MyDrive/CS4782FinalProject/results/results_plus_5.csv"

def log_result(dataset_name, accuracy):
  file_exists = os.path.isfile(file_path)

  with open(file_path, mode='a', newline='') as file:
    writer = csv.writer(file)
    if not file_exists:
      writer.writerow(["Dataset", "Accuracy"])
    writer.writerow([datatype, accuracy])

log_result(datatype, accuracy)